In [25]:
from utils_sql import create_connection_to_vector_db, get_db_table

In [26]:
# Задаем параметры для поиска поставок по wild
wild = 'wild' + str(355)
date_from = '2025-10-21'
date_to = '2025-12-13'

query_supplies = f"""SELECT supply_id, 
        product_id,
        SUM(quantity) AS tasks_count,
        DATE(created_at) AS date
    FROM inventory_transactions it
    WHERE DATE(created_at) BETWEEN '{date_from}'
    AND '{date_to}'
    AND it.delivery_type  = 'ФБС'
    AND product_id = '{wild}'
    GROUP BY supply_id, product_id, date;"""


# Устанавливаем соединение
connection = create_connection_to_vector_db()
# Делаем запрос
df_table_supllies = get_db_table(query_supplies, connection)

Соединение с БД PostgreSQL успешно установлено в 2025-12-16-11:M
Данные из БД загружены в датафрейм


In [27]:
# Отбираем уникальные поставки
supllies_unique = df_table_supllies['supply_id'].unique()
# Достаем номера поставок
supllies_num = [suplly.removeprefix('WB-GI-') for suplly in supllies_unique if suplly]
# Количество СЗ в этих поставках
tasks_count = df_table_supllies['tasks_count'].sum()
print(f"Через сервис проведено {len(supllies_unique)} поставок за период с {date_from} по {date_to}.")
print()
print(f"В которых содржится {int(tasks_count)} сборочных заданий")

Через сервис проведено 413 поставок за период с 2025-10-21 по 2025-12-13.

В которых содржится 9985 сборочных заданий


In [28]:
# Вычислим сколько поставок в этот же период было по данным ВБ
query_supplies_by_wb = f"""SELECT 
	s.closed_at,
	s.id AS supply_id, 
	s."name",
	s2.id AS task_id,
	s2.local_vendor_code,
	s.account
FROM supplies_data s 
LEFT JOIN supplies_and_orders s2
ON s.id = s2.supply_id
WHERE DATE(s.closed_at) BETWEEN '{date_from}'
AND '{date_to}'
AND s2.local_vendor_code = '{wild}';"""
df_supplies_by_wb = get_db_table(query_supplies_by_wb, connection)
# Отберем уникальные поставки, по данным ВБ
supllies_unique_wb = df_supplies_by_wb['supply_id'].unique()

# Вычислим есть ли поставки на ВБ, которые не попали в наш сервис
supply_difference = set(supllies_unique_wb)-set(supllies_unique)
if len(supply_difference) > 0:
	print(f"Найдены следующие поставки, отсутствующие в сервисе:")
	print(supply_difference)
	print()
	count_tasks_not_in_service = df_supplies_by_wb[df_supplies_by_wb['supply_id'].isin(supply_difference)]['task_id'].count()
	print(f"Таким образом мимо сервиса прошло {count_tasks_not_in_service} сборочных заданий")
else:
	print("Расхождений не обнаружено")

Данные из БД загружены в датафрейм
Найдены следующие поставки, отсутствующие в сервисе:
{'WB-GI-192798925', 'WB-GI-192799187', 'WB-GI-195760779', 'WB-GI-195760776', 'WB-GI-201416453', 'WB-GI-195760775', 'WB-GI-195760778', 'WB-GI-195760777'}

Таким образом мимо сервиса прошло 130 сборочных заданий


In [29]:
# Узнаем количество принятых в этих поставках
query_accept = f"""
SELECT *
FROM acceptance_fbs_advanced afa
WHERE afa.local_vendor_code = '{wild}'
AND afa.document_number IN {tuple(supllies_num)}
"""
df_table_accept_qnt = get_db_table(query_accept, connection)
# Количество принятых СЗ
count_accept = df_table_accept_qnt['order_number'].count()
print(f"Было заявлено {int(tasks_count)} сборочных заданий в {len(supllies_num)} поставках")
print()
print(f"Принято {count_accept} сборочных заданий по {wild} за период с {date_from} по {date_to}")
print()
print(f"Таким образом не принято {int(tasks_count-count_accept)} сборочных заданий")
print()
print(f"Что составляет {round((1-count_accept/tasks_count)*100)} % от общего количества СЗ переданных в доставку за указанный период")

Данные из БД загружены в датафрейм
Было заявлено 9985 сборочных заданий в 412 поставках

Принято 9669 сборочных заданий по wild355 за период с 2025-10-21 по 2025-12-13

Таким образом не принято 316 сборочных заданий

Что составляет 3 % от общего количества СЗ переданных в доставку за указанный период


In [ ]:
'WB-GI-201993171',
'WB-GI-202004755',
'WB-GI-202150071',
'WB-GI-201986814',
'WB-GI-201993127',
'WB-GI-201709111',
'WB-GI-201986811',
'WB-GI-201994066',
'WB-GI-201715957',
'WB-GI-201986812',
'WB-GI-201993172',
'WB-GI-202408008'



TTFError: Can't open file "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"